In [1]:
# 1. Descarga e impresión de robot.txt

import requests

headers = {"User-agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36"}

robot_gobpe=requests.get("https://www.gob.pe/robots.txt", headers=headers, timeout=30)

print(robot_gobpe.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



1.a ¿Qué prohíbe el archivo robots.txt?

Prohíbe el acceso a las rutas que incluyan la ruta /admin/ y el parámetro sheet=.

Para el caso en concreto de los bots GPT y OAI-Search se les pide, además, esperar 5 y 2 segundos entre solicitudes, respectivamente. 

1.b ¿La página de búsquedas /búsquedas está permitida?

Sí, en la medida que no incluya el parámetro sheet=.

1.c ¿Por qué conviene hacer pausas entre página y página? 

Si bien el Crawl-delay alcanza únicamente a los bots de GPT y OAI-search, parece indicar que se busca evitar que el sitio se sature con muchas solicitudes y se bloquee.

In [2]:
# 2. Búsqueda de un mes de la temporada (Enero 2025)

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as PD 
import time
import re

In [63]:
opciones = Options()
# opciones.add_argument("--headless=new")   # descomenta para ocultarlo
opciones.add_argument("--window-size=1400,900")
opciones.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")


navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2025&hasta=31-01-2025")


WebDriverWait(navegador, 30)
time.sleep(6)

Navegador abierto


In [73]:
# 3. Número total de resultados (Enero 2025)

texto_total = navegador.find_element(By.CSS_SELECTOR, "main h2").text
print(texto_total)

total_enero = int(re.search(r"\d+", texto_total).group())
print("Total de resultados:", total_enero)

12 Resultados
Total de resultados: 12


In [74]:
# 4. Extracción de número, fecha, título y enlace de cada norma (Enero 2025)

espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_pagina = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas))
print("Total según la página:", total_pagina)

Normas extraídas     : 12
Total según la página: 12


In [77]:
PD.DataFrame(normas)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 041-2025-PCM,Publicado: 26 de marzo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco",https://www.gob.pe/institucion/pcm/normas-legales/6709207-041-2025-pcm
1,Decreto Supremo N.° 040-2025-PCM,Publicado: 26 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac - Cusco - Arequipa,https://www.gob.pe/institucion/pcm/normas-legales/6709197-040-2025-pcm
2,Decreto Supremo N.° 036-2025-PCM,Publicado: 19 de marzo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...",https://www.gob.pe/institucion/pcm/normas-legales/6608813-036-2025-pcm
3,Decreto Supremo N.° 035-2025-PCM,Publicado: 17 de marzo de 2025,Decreto Supremo que declara el Estado de Emergencia en Lima Metropolitana del departamento de Lima y en la Provincia Constitucional del Callao,https://www.gob.pe/institucion/pcm/normas-legales/6609801-035-2025-pcm
4,Decreto Supremo N.° 034-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6608812-034-2025-pcm
5,Decreto Supremo N.° 033-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6608811-033-2025-pcm
6,Decreto Supremo N.° 032-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en el sector comprendido entre el Jr. Jauja, Jr. Áncash, Jr. Huánuco, Prolongación Lucanas, Jr. Santa Rosa y Jr. Huanta del Cercado de Lima de la p ...",https://www.gob.pe/institucion/pcm/normas-legales/6608810-032-2025-pcm
7,Decreto Supremo N.° 031-2025-PCM,Publicado: 6 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6595467-031-2025-pcm
8,Decreto Supremo N.° 030-2025-PCM,Publicado: 6 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6595466-030-2025-pcm
9,Decreto Supremo N.° 028-2025-PCM,Publicado: 5 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Anco y Unión Progreso de la provincia de La Mar del departamento de Ayacucho y en los distritos de Megantoni, Kumpirushiato, Ec ...",https://www.gob.pe/institucion/pcm/normas-legales/6595463-028-2025-pcm


In [78]:
# 5.1 Extracción de número, fecha, título y enlace de cada norma (Febrero 2025)

time.sleep(5)  

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-02-2025&hasta=28-02-2025")

Navegador abierto


In [79]:
espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas_febrero = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas_febrero.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_febrero = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas_febrero))
print("Total según la página:", total_febrero)

Normas extraídas     : 4
Total según la página: 4


In [80]:
PD.DataFrame(normas_febrero)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 025-2025-PCM,Publicado: 26 de febrero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes,https://www.gob.pe/institucion/pcm/normas-legales/6539147-025-2025-pcm
1,Decreto Supremo N.° 024-2025-PCM,Publicado: 26 de febrero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6539143-024-2025-pcm
2,Decreto Supremo N.° 022-2025-PCM,Publicado: 19 de febrero de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Manitea, Kimbiri y Cielo Punco de la provincia de La Convención del departamento de Cusco y en el distrito de Samuga ...",https://www.gob.pe/institucion/pcm/normas-legales/6539254-022-2025-pcm
3,Decreto Supremo N.° 021-2025-PCM,Publicado: 19 de febrero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6539250-021-2025-pcm


In [81]:
# 5.2 Extracción de número, fecha, título y enlace de cada norma (Marzo 2025)

time.sleep(5)  

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-03-2025&hasta=31-03-2025")

Navegador abierto


In [82]:
espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas_marzo = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas_marzo.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_marzo = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas_marzo))
print("Total según la página:", total_marzo)

Normas extraídas     : 12
Total según la página: 12


In [83]:
PD.DataFrame(normas_marzo)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 041-2025-PCM,Publicado: 26 de marzo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco",https://www.gob.pe/institucion/pcm/normas-legales/6709207-041-2025-pcm
1,Decreto Supremo N.° 040-2025-PCM,Publicado: 26 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac - Cusco - Arequipa,https://www.gob.pe/institucion/pcm/normas-legales/6709197-040-2025-pcm
2,Decreto Supremo N.° 036-2025-PCM,Publicado: 19 de marzo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...",https://www.gob.pe/institucion/pcm/normas-legales/6608813-036-2025-pcm
3,Decreto Supremo N.° 035-2025-PCM,Publicado: 17 de marzo de 2025,Decreto Supremo que declara el Estado de Emergencia en Lima Metropolitana del departamento de Lima y en la Provincia Constitucional del Callao,https://www.gob.pe/institucion/pcm/normas-legales/6609801-035-2025-pcm
4,Decreto Supremo N.° 034-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6608812-034-2025-pcm
5,Decreto Supremo N.° 033-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6608811-033-2025-pcm
6,Decreto Supremo N.° 032-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en el sector comprendido entre el Jr. Jauja, Jr. Áncash, Jr. Huánuco, Prolongación Lucanas, Jr. Santa Rosa y Jr. Huanta del Cercado de Lima de la p ...",https://www.gob.pe/institucion/pcm/normas-legales/6608810-032-2025-pcm
7,Decreto Supremo N.° 031-2025-PCM,Publicado: 6 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6595467-031-2025-pcm
8,Decreto Supremo N.° 030-2025-PCM,Publicado: 6 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6595466-030-2025-pcm
9,Decreto Supremo N.° 028-2025-PCM,Publicado: 5 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Anco y Unión Progreso de la provincia de La Mar del departamento de Ayacucho y en los distritos de Megantoni, Kumpirushiato, Ec ...",https://www.gob.pe/institucion/pcm/normas-legales/6595463-028-2025-pcm


In [70]:
# 6. Verificación 

verificacion = PD.DataFrame({
    "mes": ["Enero 2025", "Febrero 2025", "Marzo 2025"],
    "resultados_totales": [total_enero, total_febrero, total_marzo],
    "resultados_extraidos": [len(normas), len(normas_febrero), len(normas_marzo)],
})

print(verificacion.to_string(index=False))

         mes  resultados_totales  resultados_extraidos
  Enero 2025                  12                    12
Febrero 2025                   4                     4
  Marzo 2025                  12                    12


7. No se verifica la existencia de normas repetidas.

8. No se verifica la existencia de normas emitidas por entidades distintas de la PCM.